# Project 1: Word Embeddings / Recurrent Neural Networks

In project is part of the NLP module held in the spring of 2026. It covers two different training architectures:
- Word embeddings (word2vec, GloVe, or fastText) together with a classifier (2-layer with ReLU
non-linearity)
- A 2-layer RNN architecture (LSTM or GRU, use the PyTorch implementations), and a two-layer
classifier with a ReLU.

## Introduction

#### todo
- words and biases link
- sources
- tools

## Setup

In [143]:
# pip install datasets wandb fasttext torch nltk

In [144]:
from datasets import load_dataset
import re
import string
import ssl
import wandb
import nltk

In [145]:
wandb.login()

True

## Preprocessing

### Load Dataset

In [146]:
# Load dataset directly form parquet files because dataset scripts are no longer supported
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Explore Dataset

In [147]:
print(f"Dataset split sizes")
print(f"Train: {len(train_split)}\nValidation: {len(valid_split)}\nTest: {len(test_split)}")

print("__________________________________________________________")
print(f"Example Row: {train_split[0]}")

print("__________________________________________________________")
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'
print(f"Features: {train_split.features}\nSelected: {[COL_GOAL, COL_SOL1, COL_SOL2]}\nTarget: {COL_LABEL}")
print("__________________________________________________________")
print(f"Number of rows labeled with class '0' in train: {train_split[COL_LABEL].count(0)}")
print(f"Number of rows labeled with class '0' in total: {train_split[COL_LABEL].count(0) + valid_split[COL_LABEL].count(0) + test_split[COL_LABEL].count(0)}\n")
print(f"Number of rows labeled with class '1' in train: {train_split[COL_LABEL].count(1)}")
print(f"Number of rows labeled with class '1' in total: {train_split[COL_LABEL].count(1) + valid_split[COL_LABEL].count(1) + test_split[COL_LABEL].count(1)}")

# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print("__________________________________________________________")
print(f"Number of HTML elements found: {html_elements}")

Dataset split sizes
Train: 15113
Validation: 1000
Test: 1838
__________________________________________________________
Example Row: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
__________________________________________________________
Features: {'goal': Value('string'), 'sol1': Value('string'), 'sol2': Value('string'), 'label': ClassLabel(names=['0', '1'])}
Selected: ['goal', 'sol1', 'sol2']
Target: label
__________________________________________________________
Number of rows labeled with class '0' in train: 7536
Number of rows labeled with class '0' in total: 8963

Number of rows labeled with class '1' in train: 7577
Number of rows labeled with class '1' in total: 8988
__________________________________________________________
Number of HTML elements found: 0


### Tokenization / Punctuation Removal / Input

Stemming or Lemmatization only has to be done when the embedding model is trained with such text. 
Stopword removal would not help increasing the model performance.
Punctuation removal is done. 

In [148]:
# Skip HTTPS certificate verification to allow download
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt')
nltk.download('punkt_tab')

# Separation token is used for optimizing the hidden state of RNN
SEPARATION_TOKEN = '<SEP>'

INPUT1_FIELD = 'input1'
INPUT2_FIELD = 'input2'

def preprocess(text):
    text_lower = text.lower()
    tokens = nltk.word_tokenize(text_lower)
    tokens = [token for token in tokens if token not in string.punctuation]
    return tokens

def format_input(goal, sol):
    return goal + [SEPARATION_TOKEN] + sol

def preprocess_row(row):
    preprocessed_goal = preprocess(row[COL_GOAL])
    preprocessed_sol1 = preprocess(row[COL_SOL1])
    preprocessed_sol2 = preprocess(row[COL_SOL2])
    return {
        INPUT1_FIELD: format_input(preprocessed_goal, preprocessed_sol1),
        INPUT2_FIELD: format_input(preprocessed_goal, preprocessed_sol2)
    }

train_processed = train_split.map(preprocess_row)
valid_processed = valid_split.map(preprocess_row)
test_processed = test_split.map(preprocess_row)

print(f"Row before preprocessing: {train_split[0]}")
print(f"Processed row: {train_processed[0]}")

Row before preprocessing: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
Processed row: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1, 'input1': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<SEP>', 'pour', 'it', 'onto', 'a', 'plate'], 'input2': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<SEP>', 'pour', 'it', 'into', 'a', 'jar']}


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sachavogel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sachavogel/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Vocabulary

In [149]:
UNK_TOKEN = '<UNK>'
PAD_TOKEN = '<PAD>'

def create_vocabulary(rows):
    v = set([SEPARATION_TOKEN, UNK_TOKEN, PAD_TOKEN])
    for row in rows:
        for token in row['input1'] + row['input2']:
            v.add(token)
    return v

def create_word2idx(vocabulary):
    w2i = {}
    for i, word in enumerate(vocabulary):
        w2i[word] = i
    return w2i
            
def check_unknown(tokens):
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]
        
def replace_unknowns(row):
    return {
        INPUT1_FIELD: check_unknown(row[INPUT1_FIELD]),
        INPUT2_FIELD: check_unknown(row[INPUT2_FIELD])
    }

def count_unknowns(split):
    count = sum(token == UNK_TOKEN for row in split for token in row[INPUT1_FIELD] + row[INPUT2_FIELD])
    total = sum(len(row[INPUT1_FIELD]) + len(row[INPUT2_FIELD]) for row in split)
    return count, total
        
vocabulary = create_vocabulary(train_processed)
word2idx = create_word2idx(vocabulary)
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Example vocabulary: {list(vocabulary)[:5]}")
print(f"Word to index for first word: {word2idx[list(vocabulary)[0]]}")

valid_processed = valid_processed.map(replace_unknowns)
# test_processed = valid_processed.map(replace_unknowns)

print("__________________________________________________________")
unk_count, total_count = count_unknowns(train_processed)
print(f"{UNK_TOKEN} count in train set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")
unk_count, total_count = count_unknowns(valid_processed)
print(f"{UNK_TOKEN} count in validation set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")


Vocabulary size: 15766
Example vocabulary: ['fourths', 'pasteurizing', 'coloration', 'tier', '6-12']
Word to index for first word: 0


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

__________________________________________________________
<UNK> count in train set: 0 (0.00%)
<UNK> count in validation set: 781 (1.41%)


## Model

In [150]:
# todo

## Training

In [151]:
# todo

## Evaluation

In [152]:
# todo

## Interpretation

In [153]:
# todo